### ЗАДАЧА: Реестр абонементов фитнес-клуба

Администратор фитнес-клуба получает строки с данными об абонементах.
Нужно собрать удобную модель, которая позволит:
- загрузить клиентов в единый реестр,
- посмотреть только активные абонементы,
- отфильтровать клиентов по тарифу,
- посчитать суммарное число оставшихся посещений,
- понять, как меняется реестр после активации и списания посещения.

В данных есть абонементы с разными статусами и остатком посещений,
поэтому важно корректно валидировать тариф, статус и изменение состояния объекта.


In [ ]:
# rows: sub_id|client_name|plan|visits_left|status
rows = [
    'SB-100|Alice|standard|8|active',
    'SB-101|Bob|premium|12|frozen',
    'SB-102|Charlie|family|0|expired',
    'SB-103|Diana|standard|5|active',
]


class Subscription:
    allowed_plans = {'standard', 'premium', 'family'}
    allowed_statuses = {'active', 'frozen', 'expired'}

    def __init__(self, sub_id, client_name, plan, visits_left, status):
        # TODO: сохранить sub_id, client_name, plan
        self.sub_id = sub_id
        self.client_name = client_name
        # TODO: visits_left хранить через self._visits_left
        self._visits_left = 0
        # TODO: значение visits_left пропустить через property/setter
        self.visits_left = visits_left
        # TODO: проверить plan и status, иначе raise ValueError
        if plan not in self.allowed_plans:
            raise ValueError("Неккоректный план")
        self._plan = plan
        if status not in self.allowed_statuses:
            raise ValueError("Неккоректный статус")
        self._status = status

    @property
    def visits_left(self):
        # TODO: вернуть текущее число посещений
        return self._visits_left

    @visits_left.setter
    def visits_left(self, value):
        # TODO: привести value к int
        value = int(value)
        # TODO: если value < 0 -> raise ValueError('Visits must be >= 0')
        if value < 0:
            raise ValueError ("Количество посещений должно быть >= 0")
        # TODO: сохранить результат в self._visits_left
        self._visits_left = value

    def use_visit(self):
        # TODO: если статус не 'active' -> raise ValueError
        if self._status != 'active':
            raise ValueError("Неккоректный статус")
        # TODO: если visits_left == 0 -> raise ValueError
        if self._visits_left == 0:
            raise ValueError("Количество посещений должно быть > 0")
        # TODO: уменьшить visits_left на 1
        self._visits_left -= 1
        # TODO: если после списания visits_left == 0, перевести статус в 'expired'
        if self._visits_left == 0:
            self._status = 'expired'

    def freeze(self):
        # TODO: если статус 'expired' -> raise ValueError
        # TODO: перевести абонемент в 'frozen'
        if self.status == 'expired':
            raise ValueError("Абонемент истек")
        self._status = 'frozen'

    def activate(self):
        # TODO: если visits_left == 0 -> raise ValueError
        if self._visits_left == 0:
            raise ValueError("Активировать абонемент нельзя")
        # TODO: перевести абонемент в 'active'
        self._status = 'activate'

    @classmethod
    def from_row(cls, row):
        # TODO: split по '|'
        parts = row.split("|")
        if len(parts) != 5:
            raise ValueError("Неверный формат")
        # TODO: ожидать 5 частей: sub_id, client_name, plan, visits_left, status
        sub_id, client_name, plan, visits_left, status = parts
        # TODO: вернуть Subscription(...)
        return Subscription(sub_id, client_name, plan, visits_left, status )

    def __repr__(self):
        # TODO: вернуть строку вида Subscription(sub_id='...', client_name='...', status='...')
        return f"Subscription(sub_id='{self.sub_id}', client_name='{self.client_name}', status='{self._status}')"


class SubscriptionRegistry:
    def __init__(self):
        self.items = []

    def add(self, subscription):
        # TODO: добавить subscription в self.items
        self.items.append(subscription)

    def load(self, rows):
        # TODO: для каждой строки создать Subscription.from_row(row)
        # TODO: добавить объект в реестр через add(...)
        for row in rows:
            subscription = Subscription.from_row(row)
            self.add(subscription)

    def active_subscriptions(self):
        # TODO: вернуть список абонементов со статусом 'active'
        return [subscription for subscription in self.items if subscription._status == 'active']

    def by_plan(self, plan):
        # TODO: вернуть список абонементов нужного тарифа
        return [subscription for subscription in self.items if subscription._plan == plan]

    def total_visits_left(self):
        # TODO: вернуть суммарное число оставшихся посещений
        return sum(subscription.visits_left for subscription in self.items)

    def status_summary(self):
        # TODO: собрать dict вида status -> count
        summary = {}
        for subscription in self.items:
            status = subscription._status
            summary[status] = summary.get(status, 0) + 1
        return summary

    def find(self, sub_id):
        # TODO: вернуть абонемент по sub_id или None
        for subscription in self.items:
            if subscription.sub_id == sub_id:
                return subscription
        return None


registry = SubscriptionRegistry()

# TODO: загрузить rows в registry
registry.load(rows)
# TODO: вывести все абонементы
print("Все абонементы: ")
for subscription in registry.items:
    print(subscription)
# TODO: вывести active_subscriptions()
print("Активные абонементы: ")
for subscription in registry.active_subscriptions():
    print(subscription)
# TODO: вывести by_plan('standard')
print("Абонементы тарифа 'standard': ")
for subscription in registry.by_plan("standard"):
    print(subscription)
# TODO: вывести total_visits_left()
print("Общее количество оставшихся посещений: ", registry.total_visits_left())
# TODO: вывести status_summary()
print("Сводка по статусам: ", registry.status_summary())
# TODO: найти абонемент 'SB-101', активировать его и вывести status_summary()
bob_subscription = registry.find('SB-101')
if bob_subscription:
    bob_subscription.activate()
    print(f"После активации SB-101: {registry.status_summary()}")
# TODO: найти абонемент 'SB-100', списать одно посещение и вывести объект
alice_subscription = registry.find('SB-100')
if alice_subscription:
    alice_subscription.use_visit()
    print(f"После списания посещения для SB-100: {alice_subscription}")